## Import libraries

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import pandas as pd
import shapely
from shapely import wkt
import geopandas as gpd

import utils.labels as Labels
import utils.ahn_fuser as ahn_fuser
import utils.bgt_fuser as bgt_fuser
import utils.pipeline as Pipeline
from config import TILE_DIR, AHN_NPZ_DIR, BGT_DIR, BBOX_DIR, LABELED_DIR, SETUP_TILECODES

In [ ]:
## relevant BGT layers

tilecodes = SETUP_TILECODES

input_dir        = TILE_DIR
dir_pc_in_file   = str(TILE_DIR) + "/"
dir_pc_out_file  = str(LABELED_DIR / "road_ground_labeled_")

# Voxel downsampling size used in notebook 5 — must match DOWNSAMPLE_VOXEL there.
# min_comp_size is calculated from this: density = 1/DOWNSAMPLE_VOXEL² pts/m²,
# so pts per 0.4m grid cell ≈ (0.4/DOWNSAMPLE_VOXEL)².
# Original pipeline used min_comp_size=50 tuned for ~2600 pts/m² (raw 2024 cloud).
DOWNSAMPLE_VOXEL = 0.15   # metres — keep in sync with notebook 5
MIN_COMP_SIZE = max(2, int((0.4 / DOWNSAMPLE_VOXEL) ** 2) - 1)
print(f"min_comp_size set to {MIN_COMP_SIZE}  "
      f"(density after {DOWNSAMPLE_VOXEL}m downsample: "
      f"~{(0.4/DOWNSAMPLE_VOXEL)**2:.0f} pts per 0.4m cell)")

bgt_layers = ['BGT_WGL_rijbaan_lokale_weg', 'BGT_WGL_rijbaan_regionale_weg',
                    'BGT_WGL_rijbaan_autoweg', 'BGT_WGL_rijbaan_autosnelweg',
                    'BGT_WGL_parkeervlak', 'BGT_WGL_ov-baan', 'BGT_WGL_fietspad']

base_path = BGT_DIR

In [ ]:
df_bgt = pd.concat(
    [
        pd.read_csv(base_path / f"{file}.csv", sep=';', usecols=['bgt-functie', 'geometrie'])
        for file in bgt_layers
    ],
    ignore_index=True
)

In [ ]:
df_bgt = df_bgt.rename(columns={'geometrie':'geometry'})

df_bgt['geometry'] = df_bgt['geometry'].apply(shapely.wkt.loads)
gdf_bgt = gpd.GeoDataFrame(df_bgt, geometry='geometry' ,crs='EPSG:28992')

In [ ]:
bbox_polygon_dump = []

for tilecode in tilecodes:
    bbox_polygon = gpd.read_file(BBOX_DIR / f"bbox_{tilecode}.geojson")
    bbox_polygon_dump.append(bbox_polygon)
bbox_polygons = pd.concat(bbox_polygon_dump, ignore_index=True)

In [ ]:
## subset geodataframe 

gdf_bgt_tiles = {}
for tilecode in tilecodes:

    bbox_polygon = bbox_polygons[bbox_polygons['tilecode']==tilecode]
    gdf_bgt_tiles[tilecode] = gpd.sjoin(gdf_bgt, bbox_polygon, how="inner", predicate="intersects")

In [ ]:
for tilecode in tilecodes:

    # Find LAZ file recursively (tiles may be in subfolders)
    matches = list(Path(input_dir).rglob(f"{tilecode}.laz")) + \
              list(Path(input_dir).rglob(f"{tilecode}.LAZ"))
    if not matches:
        print(f"  [skip] {tilecode} — LAZ file not found under {input_dir}")
        continue
    in_file = str(matches[0])

    # Ground fuser
    npz_ground_fuser = ahn_fuser.NPZAHNFuser(
        label=Labels.Labels.GROUND,
        npz_reader=str(AHN_NPZ_DIR / f"ahn_{tilecode}.npz"),
        target='ground',
        epsilon=0.2,
        grid_size=0.4,
        min_comp_size=MIN_COMP_SIZE,   # scaled to downsampled density
    )

    npz_building_fuser = ahn_fuser.NPZAHNFuser(
        label=Labels.Labels.BUILDING,
        npz_reader=str(AHN_NPZ_DIR / f"ahn_{tilecode}.npz"),
        target='building',
        epsilon=0.2,
        min_comp_size=MIN_COMP_SIZE,   # scaled to downsampled density
    )

    # Road fuser using GeoDataFrame
    road_part_fuser = bgt_fuser.BGTRoadFuser(
        label=Labels.Labels.ROAD,
        bgt_gdf=gdf_bgt_tiles[tilecode],
        bgt_types=None,
        offset=0
    )

    # Pipeline
    process_sequence = (npz_ground_fuser, npz_building_fuser, road_part_fuser)
    pipeline = Pipeline.Pipeline(processors=process_sequence, caching=False)

    out_file = f"{dir_pc_out_file}{tilecode}.laz"
    pipeline.process_file(in_file, out_file=out_file)

In [ ]:
# ## Faster apporach - to check ?

# from multiprocessing import Pool
# import os

# def process_tile(tilecode):

#     npz_ground_fuser = ahn_fuser.NPZAHNFuser(
#         label=Labels.Labels.GROUND,
#         npz_reader=f"data/output/ahn/ahn_{tilecode}.npz",
#         target='ground',
#         epsilon=0.2,
#         grid_size=0.4,
#         min_comp_size=50
#     )

#     npz_building_fuser = ahn_fuser.NPZAHNFuser(
#         label=Labels.Labels.BUILDING,
#         npz_reader=f"data/output/ahn/ahn_{tilecode}.npz",
#         target='building',
#         epsilon=0.2
#     )

#     road_part_fuser = bgt_fuser.BGTRoadFuser(
#         label=Labels.Labels.ROAD,
#         bgt_gdf=gdf_bgt_tiles[tilecode],
#         bgt_types=None,
#         offset=0
#     )

#     pc_in_file = f"data/input/pointcloud/raw/{tilecode}.laz"
#     pc_out_file = f"data/output/road_ground_labeled_{tilecode}.laz"

#     pipeline = Pipeline.Pipeline(
#         processors=(npz_ground_fuser, npz_building_fuser, road_part_fuser),
#         caching=False
#     )

#     pipeline.process_file(pc_in_file, out_file=pc_out_file)


# if __name__ == "__main__":

#     with Pool(os.cpu_count()) as pool:
#         pool.map(process_tile, tilecodes)

## Diagnostic — Raw 2025 tile (downsampled)

Run the same pipeline on the full 2025 tile (downsampled to 0.15 m, no stability filter).
Compare output with the stability-filtered `yearly_filtered/120300_489300.laz` to diagnose
whether road-labeling failures are caused by the stability filter or are tile-specific.

In [ ]:
# Input: raw 2025 tile, downsampled to 0.15 m (exported by notebook 5)
# Output: separate file so it doesn't overwrite the stability-filtered result
dir_pc_in_raw2025  = str(TILE_DIR.parent / "yearly_filtered") + "/"
dir_pc_out_raw2025 = str(LABELED_DIR / "2025_road_ground_labeled_raw2025_")

tilecodes_diag = SETUP_TILECODES

for tilecode in tilecodes_diag:
    in_file  = f"{dir_pc_in_raw2025}{tilecode}_2025_raw.laz"
    out_file = f"{dir_pc_out_raw2025}{tilecode}.laz"

    npz_ground_fuser = ahn_fuser.NPZAHNFuser(
        label=Labels.Labels.GROUND,
        npz_reader=str(AHN_NPZ_DIR / f"ahn_{tilecode}.npz"),
        target='ground',
        epsilon=0.2,
        grid_size=0.4,
        min_comp_size=MIN_COMP_SIZE,
    )
    npz_building_fuser = ahn_fuser.NPZAHNFuser(
        label=Labels.Labels.BUILDING,
        npz_reader=str(AHN_NPZ_DIR / f"ahn_{tilecode}.npz"),
        target='building',
        epsilon=0.2,
        min_comp_size=MIN_COMP_SIZE,
    )
    road_part_fuser = bgt_fuser.BGTRoadFuser(
        label=Labels.Labels.ROAD,
        bgt_gdf=gdf_bgt_tiles[tilecode],
        bgt_types=None,
        offset=0
    )
    process_sequence = (npz_ground_fuser, npz_building_fuser, road_part_fuser)
    pipeline = Pipeline.Pipeline(processors=process_sequence, caching=False)

    print(f"Processing raw 2025 tile: {in_file}")
    print(f"  → {out_file}")
    pipeline.process_file(in_file, out_file=out_file)

## Optional plotting

In [ ]:
import laspy
import numpy as np

In [ ]:
# Read the processed LAS/LAZ
pc = laspy.read("data/output/labeled_pointcloud/road_ground_labeled_" + tilecode + '.laz')

points = np.vstack((pc.x, pc.y, pc.z)).T

# Labels (usually stored in 'label' extra dimension)
if "label" in pc.point_format.extra_dimension_names:
    labels = pc.label
else:
    # fallback
    labels = np.zeros(len(points), dtype=np.uint16)

print("Number of points:", len(points))
print("Label counts:", np.unique(labels, return_counts=True))

In [ ]:
import numpy as np
import pandas as pd
import datashader as ds
import datashader.transfer_functions as tf

# Convert to memory-efficient types
x = np.asarray(pc.x, dtype=np.float32)
y = np.asarray(pc.y, dtype=np.float32)
labels = np.asarray(pc.label, dtype=np.int16)

df = pd.DataFrame({
    "x": x,
    "y": y,
    "label": labels
})

# 🔥 REQUIRED for count_cat
df["label"] = df["label"].astype("category")

cvs = ds.Canvas(plot_width=1200, plot_height=1200)
agg = cvs.points(df, "x", "y", ds.count_cat("label"))

color_key = {
    0: "gray",      # Unknown
    1: "red",       # Road
    9: "white",     # Ground
    10: "blue",     # Building
    30: "green",    #Tree
    40: "orange",   #Car
    60: "yellow",   #lamp post
    83: "purple"

}

img = tf.shade(agg, color_key=color_key)
img